In [ ]:
import jax
import jax.numpy as jnp
from jax import random, grad, vmap
from jax.tree import map
from functools import partial
import optax
from flax.training import train_state
from flax import linen as nn
from jax.scipy.integrate import trapezoid

In [2]:
# Define a simple MLP using Flax Linen
class MLP(nn.Module):
    architecture: list
    hidden_activation: callable = nn.tanh

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = nn.Dense(features=self.architecture[i + 1])(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)
        return x

In [ ]:
def local_energy(x,params, model_apply): 
    psi = model_apply(params, x) # So that we don't compute twice the wavefunction, we could compute this with the log_psi
    kinetic = -0.5 * jnp.gradient(jnp.gradient(psi, x),x)
    potential = 0.5 * x**2
    return kinetic/psi + potential

def local_energy_batch(params, xs, model_apply, debug=False):
    def psi_fn(x):
        return model_apply(params, x)

    # get ψ(x) for the whole batch
    psi = jax.vmap(psi_fn)(xs)
    if debug:
        print(xs.shape)
        print(psi.shape)
        print(type(psi))

    # first and second derivative wrt x for the whole batch
    dpsi_dx = jnp.gradient(psi, xs.squeeze(), axis=0)
    d2psi_dx2 = jnp.gradient(dpsi_dx, xs.squeeze(), axis=0)

    kinetic = -0.5 * d2psi_dx2 / psi
    potential = (0.5 * xs**2).reshape(-1,1)
    if debug:
        print(kinetic.shape)
        print(potential.shape)
    return kinetic + potential

def log_psi(x, params, model_apply):
    psi = model_apply(params, x)
    return jnp.log(jnp.abs(psi) + 1e-8).squeeze()  # Add small constant to avoid log(0)

grad_log_psi = jax.grad(lambda params, x, model_apply: log_psi(x, params, model_apply), argnums=0) # CAREFUL WITH THE DERIVATIVES: WE WANT THE GRADIENT WRT THE PARAMETERS, so we turn the order of the p and x

# now grad_log_psi is a function that takes (params, x, model_apply) and returns the gradient of log_psi wrt params

def log_psi_and_grad(params, x, model_apply):
    psi = model_apply(params, x)
    log_psi_val = jnp.log(jnp.abs(psi) + 1e-8)
    grad_log_psi_val = grad_log_psi(params, x, model_apply)
    return log_psi_val, grad_log_psi_val

def energy_fn(params, batch, model_apply):
    psi = jax.vmap(lambda x: model_apply(params, x))(batch)
    psi_squared = jnp.abs(psi)**2
    local_energy_per_point = local_energy_batch(params, batch, model_apply)

    E = jnp.mean(local_energy_per_point)
    return E

def loss_and_grads(params, batch, model_apply):
    local_energy_per_point = local_energy_batch(params, batch, model_apply)
    E = energy_fn(params, batch, model_apply)
    E_centered = local_energy_per_point - E
    log_psi_grads = jax.vmap(lambda x: grad_log_psi(params, x, model_apply))(batch)
    print(log_psi_grads)
    # grad_E = 2 * jnp.mean(E_centered * log_psi_grads, axis=0)
    grad_E = jax.tree_util.tree_map(lambda g: 2 *jnp.mean(E_centered[:, None] * g, axis=0), log_psi_grads)
    print(grad_E)
    return E, grad_E

@jax.jit
def train_step(state, batch):
    E, grads = loss_and_grads(state.params, batch, state.apply_fn)
    new_state = state.apply_gradients(grads=grads)
    return new_state, E

@partial(jax.jit, static_argnums=(1,))
def mh_kernel(rng_key, prob_fn, prob_params, position, prob, PBC=None):
    key1, key2 = random.split(rng_key)
    proposal = position + random.normal(key1, shape=position.shape)
    # ensure PBC
    proposal = jax.lax.cond(
    PBC is None,
    lambda p: p,
    lambda p: ((p + 0.5 * PBC) % PBC) - 0.5 * PBC,
    proposal
    )
    proposal_prob = prob_fn(proposal, prob_params)
    accept = jax.random.uniform(key2) < (proposal_prob / prob)
    new_position = jnp.where(accept, proposal, position)
    new_prob = jnp.where(accept, proposal_prob, prob)
    return new_position, new_prob


@partial(jax.jit, static_argnums=(1, 2))
def mh_chain(rng_key, n_steps, prob_fn, prob_params, init_position):
    def body_fn(i, val):
        key, position, prob = val
        _, key = random.split(key)
        new_position, new_prob = mh_kernel(
            key, prob_fn, prob_params, position, prob, PBC=5.0
        )
        return key, new_position, new_prob

    try:
        init_prob = prob_fn(init_position, prob_params)
    except Exception as e:
        print("Error in prob_fn with init_position shape:", init_position.shape)
        import numpy as np
        print("trying to reshape")
        init_position = jnp.atleast_1d(init_position).reshape(-1, 1)
        init_prob = prob_fn(init_position, prob_params)
    init_val = (rng_key, init_position, init_prob)
    _, positions, _ = jax.lax.fori_loop(0, n_steps, body_fn, init_val)
    return positions

def train(n_steps, init_params, model_apply, optimizer):

    state = train_state.TrainState.create(
        apply_fn=model_apply,
        params=init_params,
        tx=optimizer
    )

    def prob_fn(x, params):
        # Ensure x has a batch dimension, run the MLP, and return non-negative per-input values.
        x = jnp.atleast_1d(x).reshape(-1, 1)  # (batch, 1)
        forward = model_apply(params, x).flatten()  # (batch,)
        out = jnp.square(forward)  # non-negative density proxy
        return jnp.squeeze(out)  # scalar for scalar input, (batch,) for batch

    energy_history = []
    batch = jnp.linspace(-5,5,1000).reshape(-1,1)
    sampler = jax.vmap(mh_chain, in_axes=(0, None, None, None, 0), out_axes=0)
    n_chains = 10_000
    rng_keys = random.split(random.PRNGKey(42), n_chains)
    init_position = jnp.zeros(n_chains)

    for step in range(n_steps):
        batch = sampler(rng_keys, n_steps, prob_fn, state.params, init_position)
        batch = batch.reshape(-1,1)  # Flatten the chains into a single batch
        state, energy = train_step(state, batch)
        energy_history.append(energy)

        if step % 100 == 0:
            print(f"Step {step}, Energy: {energy}")

    return state.params, energy_history

In [35]:
model = MLP(architecture=[1, 30, 1])
rng = jax.random.PRNGKey(0)
input_shape = (1000, 1)  # Batch size of 1000, input dimension
params = model.init(rng, jnp.ones(input_shape))  # Initialize parameters
params_fin, energy = train(1000, params, model.apply, optax.adam(1e-3))
import matplotlib.pyplot as plt
plt.plot(energy)
plt.show()

# Reconstruct wavefunction
x = jnp.linspace(-5,5,1000).reshape(-1,1)
psi_approx = model.apply(params_fin, x)
print(type(psi_approx))
print(psi_approx.shape)
norm = jnp.sqrt(trapezoid((psi_approx**2).squeeze(), x.squeeze()))
print(f"Norm: {norm}")
psi_approx = psi_approx / norm
print(f"Norm after normalization: {jnp.sqrt(trapezoid((psi_approx**2).squeeze(), x.squeeze()))}")
plt.plot(x, psi_approx**2)
plt.plot(x, jnp.pi**(-0.5)*jnp.exp(-x**2), linestyle='dashed')

{'params': {'Dense_0': {'bias': JitTracer<float32[10000,30]>, 'kernel': JitTracer<float32[10000,1,30]>}, 'Dense_1': {'bias': JitTracer<float32[10000,1]>, 'kernel': JitTracer<float32[10000,30,1]>}}}
{'params': {'Dense_0': {'bias': JitTracer<float32[10000,30]>, 'kernel': JitTracer<float32[1,30]>}, 'Dense_1': {'bias': JitTracer<float32[10000,1]>, 'kernel': JitTracer<float32[30,1]>}}}
Step 0, Energy: 1.6501210927963257
Error in prob_fn with init_position shape: ()


TracerArrayConversionError: The numpy.ndarray conversion method __array__() was called on traced array with shape float32[]
The error occurred while tracing the function mh_chain at /tmp/ipykernel_18086/3166069848.py:86 for jit. This concrete value was not available in Python because it depends on the value of the argument init_position.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerArrayConversionError